## --------------------------------

## CtU-13 Prepration

In [24]:
import os
import pandas as pd

# Directory containing the .flow files
directory = 'CTU-Experiment/CTU_13/10_Rbot/'

# List to store individual dataframes
dataframes = []

# Loop through each file in the directory
for filename in os.listdir(directory):
    if filename.endswith('.flow'):
        # Read the file into a dataframe
        df = pd.read_csv(os.path.join(directory, filename))  # Adjust the reading method if necessary
        #df['Label'] = df['Label'].apply(lambda x: 1 if 'Botnet' in x else (0 if 'Normal' in x or 'Background' in x else x))
        df['Label'] = df['Label'].apply(lambda x: 1 if 'Botnet' in x else (0 if 'Normal' in x else x))
        df.rename(columns={'Label': 'label'}, inplace=True)

        # Remove rows where 'label' contains 'background'
        df = df[~df['label'].str.contains('Background', case=False, na=False)]

        # Append the dataframe to the list
        dataframes.append(df)


# Concatenate all dataframes into a single dataframe
combined_df = pd.concat(dataframes, ignore_index=True)

# Display the combined dataframe
print(combined_df)

                         StartTime        SrcAddr         DstAddr Proto  \
0       2011/08/18 12:49:15.693956  147.32.84.164  74.125.232.215   tcp   
1       2011/08/18 12:49:18.143576  147.32.84.164  74.125.232.197   tcp   
2       2011/08/18 12:49:18.303590  147.32.84.164  209.85.149.138   tcp   
3       2011/08/18 12:49:19.838272  147.32.84.170     147.32.80.9   udp   
4       2011/08/18 12:49:19.839123  147.32.84.170     147.32.80.9   udp   
...                            ...            ...             ...   ...   
122194  2011/08/18 17:34:58.033603  147.32.84.164    147.32.96.69  icmp   
122195  2011/08/18 17:34:59.032685  147.32.84.164    147.32.96.69  icmp   
122196  2011/08/18 17:34:59.532629  147.32.84.170     147.32.80.9   udp   
122197  2011/08/18 17:34:59.533932  147.32.84.170     147.32.80.9   udp   
122198  2011/08/18 17:34:59.535652  147.32.84.170  195.24.232.205   tcp   

         Sport   Dport          Dur  TotBytes  SrcBytes  DstBytes  ...  \
0        54784     443  3

In [25]:
import os
import yaml
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
import re

def keep_features(df, features_to_keep):
    """
    Drop all columns from the DataFrame except for the specified features.
    
    Parameters:
    - df: pd.DataFrame, the input DataFrame
    - features_to_keep: list, list of column names to retain
    
    Returns:
    - pd.DataFrame with only the specified columns
    """
    # Ensure that the features_to_keep are in the DataFrame
    features_to_keep = [feature for feature in features_to_keep if feature in df.columns]
    
    # Return a DataFrame with only the specified features
    return df[features_to_keep]

# Function to preprocess each dataset
def preprocess_dataset(df):
    # Drop columns that contain only missing values
    df = df.dropna(axis=1, how='all')
    # Separate numeric and non-numeric columns
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    non_numeric_cols = df.select_dtypes(exclude=[np.number]).columns
    
    # Check if DataFrame has either numeric or non-numeric columns
    if not numeric_cols.empty or not non_numeric_cols.empty:
        # Handle missing values for numeric data
        if not numeric_cols.empty:
            imputer_numeric = SimpleImputer(strategy='mean')
            df_numeric = pd.DataFrame(imputer_numeric.fit_transform(df[numeric_cols]), columns=numeric_cols)
        else:
            df_numeric = pd.DataFrame()  # Empty DataFrame for numeric data if no numeric columns exist
        
        # Concatenate processed numeric and encoded non-numeric data
        df_preprocessed = pd.concat([df_numeric], axis=1)
        
        features = [
            'pRetran', 'Max,sMeanPktSz', 'SrcRetra', 'PCRatio', 
            'SrcWin', 'SrcLoss', 'DstRate', 'SrcLoad', 'Load', 'DstLoad', 'TcpRtt', 'Sum', 'AckDat', 'dTtl', 'Min', 'pLoss', 'DstLoss',
            'Loss', 'StdDev', 'Rate', 'SrcRate', 'IdleTime', 'Dur', 'SrcPkts',
            'SrcGap', 'DstBytes', 'DstGap', 'sTtl', 'DstWin', 'TotPkts', 'DstPkts', 'Mean',
            'SrcBytes', 'TotBytes', 'dMeanPktSz', 'DstRetra', 'SynAck'
        ]

        # Drop all columns except the ones in features_to_keep
        df_preprocessed = keep_features(df_preprocessed, features)  

        # Scale features
        scaler = MinMaxScaler()
        df_scaled = pd.DataFrame(scaler.fit_transform(df_preprocessed), columns=df_preprocessed.columns)

        return df_scaled
    else:
        return pd.DataFrame()
    
# Step 1: Separate the label column
X = combined_df.drop(columns=['label'])
y = combined_df['label']

pre_df = preprocess_dataset(X)

pre_df['label'] = y

print(pre_df)

pre_df.to_csv('DATASETS/CTU-13/10_rbot.csv', index=False)

        pRetran  SrcRetra   PCRatio    SrcWin  SrcLoss       DstRate  \
0           0.0       0.0  0.198215  0.000131      0.0  9.287640e-07   
1           0.0       0.0  0.500000  0.000016      0.0  6.058600e-08   
2           0.0       0.0  0.500000  0.000021      0.0  6.063600e-08   
3           0.0       0.0  0.101266  0.010606      0.0  0.000000e+00   
4           0.0       0.0  0.101266  0.010606      0.0  0.000000e+00   
...         ...       ...       ...       ...      ...           ...   
122194      0.0       0.0  0.500000  0.010606      0.0  0.000000e+00   
122195      0.0       0.0  0.500000  0.010606      0.0  0.000000e+00   
122196      0.0       0.0  0.243750  0.010606      0.0  0.000000e+00   
122197      0.0       0.0  0.198979  0.010606      0.0  0.000000e+00   
122198      0.0       0.0  1.000000  0.001799      0.0  2.668624e-05   

             SrcLoad          Load       DstLoad    TcpRtt  ...    DstWin  \
0       5.280485e-06  2.210595e-05  6.191215e-06  0.000000

## -------------------------------

## ISCX VPN & TOR Prepration

In [4]:
import os
import pandas as pd

# Directory containing the .flow files
directory = 'ISCX-Experiment/ISCX_VPN/scale_5/'

# List to store individual dataframes
dataframes = []

# Loop through each file in the directory
for filename in os.listdir(directory):
    if filename.endswith('.flow'):
        # Read the file into a dataframe
        df = pd.read_csv(os.path.join(directory, filename), low_memory=False)  # Adjust the reading method if necessary
        
        # Add a label column with the filename (without extension) as the label value
        df['label'] = os.path.splitext(filename)[0]
        
        # Append the dataframe to the list
        dataframes.append(df)

# Concatenate all dataframes into a single dataframe
combined_df = pd.concat(dataframes, ignore_index=True)

# Display the combined dataframe
print(combined_df)

       TotBytes  SrcBytes  DstBytes  SrcGap  DstGap  sMeanPktSz  dMeanPktSz  \
0           782       662       120     0.0     0.0  220.666672        40.0   
1           202       162        40     0.0     0.0  162.000000        40.0   
2           105        41        64     0.0     0.0   41.000000        64.0   
3           516       476        40     0.0     0.0  476.000000        40.0   
4           364       162       202     0.0     0.0  162.000000       101.0   
...         ...       ...       ...     ...     ...         ...         ...   
44195     62625     31375     31250     NaN     NaN  125.000000       125.0   
44196     62375     31125     31250     NaN     NaN  125.000000       125.0   
44197     62750     31375     31375     NaN     NaN  125.000000       125.0   
44198     62375     31125     31250     NaN     NaN  125.000000       125.0   
44199     44875     22375     22500     NaN     NaN  125.000000       125.0   

       sMaxPktSz  dMaxPktSz  sMinPktSz  ...  PCRati

In [5]:
import os
import yaml
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
import re

def keep_features(df, features_to_keep):
    """
    Drop all columns from the DataFrame except for the specified features.
    
    Parameters:
    - df: pd.DataFrame, the input DataFrame
    - features_to_keep: list, list of column names to retain
    
    Returns:
    - pd.DataFrame with only the specified columns
    """
    # Ensure that the features_to_keep are in the DataFrame
    features_to_keep = [feature for feature in features_to_keep if feature in df.columns]
    
    # Return a DataFrame with only the specified features
    return df[features_to_keep]

# Function to preprocess each dataset
def preprocess_dataset(df):
    # Drop columns that contain only missing values
    df = df.dropna(axis=1, how='all')
    # Separate numeric and non-numeric columns
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    non_numeric_cols = df.select_dtypes(exclude=[np.number]).columns
    
    # Check if DataFrame has either numeric or non-numeric columns
    if not numeric_cols.empty or not non_numeric_cols.empty:
        # Handle missing values for numeric data
        if not numeric_cols.empty:
            imputer_numeric = SimpleImputer(strategy='mean')
            df_numeric = pd.DataFrame(imputer_numeric.fit_transform(df[numeric_cols]), columns=numeric_cols)
        else:
            df_numeric = pd.DataFrame()  # Empty DataFrame for numeric data if no numeric columns exist
        
        # Concatenate processed numeric and encoded non-numeric data
        df_preprocessed = pd.concat([df_numeric], axis=1)
        
        features = [
            'pRetran', 'Max,sMeanPktSz', 'SrcRetra', 'PCRatio', 
            'SrcWin', 'SrcLoss', 'DstRate', 'SrcLoad', 'Load', 'DstLoad', 'TcpRtt', 'Sum', 'AckDat', 'dTtl', 'Min', 'pLoss', 'DstLoss',
            'Loss', 'StdDev', 'Rate', 'SrcRate', 'IdleTime', 'Dur', 'SrcPkts',
            'SrcGap', 'DstBytes', 'DstGap', 'sTtl', 'DstWin', 'TotPkts', 'DstPkts', 'Mean',
            'SrcBytes', 'TotBytes', 'dMeanPktSz', 'DstRetra', 'SynAck'
        ]

        # Drop all columns except the ones in features_to_keep
        df_preprocessed = keep_features(df_preprocessed, features)  

        # Scale features
        #scaler = MinMaxScaler()
        #df_scaled = pd.DataFrame(scaler.fit_transform(df_preprocessed), columns=df_preprocessed.columns)

        #return df_scaled
        return df_preprocessed
    else:
        return pd.DataFrame()
    
# Step 1: Separate the label column
X = combined_df.drop(columns=['label'])
y = combined_df['label']

pre_df = preprocess_dataset(X)

pre_df['label'] = y

print(pre_df)

pre_df.to_csv('DATASETS/ISCX_VPN/ISCX_VPN.csv', index=False)

       pRetran  SrcRetra  PCRatio         SrcWin  SrcLoss    DstRate  \
0          0.0       0.0     -0.0   16384.000000      0.0   2.376705   
1          0.0       0.0     -0.0   16384.000000      0.0   0.000000   
2          0.0       0.0     -0.0   64931.000000      0.0   0.000000   
3          0.0       0.0     -0.0   16384.000000      0.0   0.000000   
4          0.0       0.0     -0.0   16384.000000      0.0   0.204698   
...        ...       ...      ...            ...      ...        ...   
44195      0.0       0.0     -0.0  171785.238582      0.0  49.801186   
44196      0.0       0.0     -0.0  171785.238582      0.0  50.082535   
44197      0.0       0.0     -0.0  171785.238582      0.0  50.005039   
44198      0.0       0.0     -0.0  171785.238582      0.0  49.959949   
44199      0.0       0.0     -0.0  171785.238582      0.0  50.068111   

            SrcLoad           Load       DstLoad  TcpRtt  ...         DstWin  \
0       4202.015137    4962.561035    760.545715     0.

OSError: Cannot save file into a non-existent directory: 'DATASETS\ISCX_VPN'

## -------------------------------

## CDR-MLC Prepration

In [4]:
import os
import yaml
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
import re

def keep_features(df, features_to_keep):
    """
    Drop all columns from the DataFrame except for the specified features.
    
    Parameters:
    - df: pd.DataFrame, the input DataFrame
    - features_to_keep: list, list of column names to retain
    
    Returns:
    - pd.DataFrame with only the specified columns
    """
    # Ensure that the features_to_keep are in the DataFrame
    features_to_keep = [feature for feature in features_to_keep if feature in df.columns]
    
    # Return a DataFrame with only the specified features
    return df[features_to_keep]

# Function to preprocess each dataset
def preprocess_dataset(df):
    # Drop columns that contain only missing values
    df = df.dropna(axis=1, how='all')
    # Separate numeric and non-numeric columns
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    non_numeric_cols = df.select_dtypes(exclude=[np.number]).columns
    
    # Check if DataFrame has either numeric or non-numeric columns
    if not numeric_cols.empty or not non_numeric_cols.empty:
        # Handle missing values for numeric data
        if not numeric_cols.empty:
            imputer_numeric = SimpleImputer(strategy='mean')
            df_numeric = pd.DataFrame(imputer_numeric.fit_transform(df[numeric_cols]), columns=numeric_cols)
        else:
            df_numeric = pd.DataFrame()  # Empty DataFrame for numeric data if no numeric columns exist
        
        # Concatenate processed numeric and encoded non-numeric data
        df_preprocessed = pd.concat([df_numeric], axis=1)
        
        features = [
            'pRetran','sMeanPktSz', 'SrcRetra', 'PCRatio', 
            'SrcWin', 'SrcLoss', 'DstRate', 'SrcLoad', 'Load', 'DstLoad', 'TcpRtt', 'Sum', 'AckDat', 'dTtl', 'Min', 'Max', 'pLoss', 'DstLoss',
            'Loss', 'StdDev', 'Rate', 'SrcRate', 'IdleTime', 'Dur', 'SrcPkts',
            'SrcGap', 'DstBytes', 'DstGap', 'sTtl', 'DstWin', 'TotPkts', 'DstPkts', 'Mean',
            'SrcBytes', 'TotBytes', 'dMeanPktSz', 'DstRetra', 'SynAck'
        ]

        # Drop all columns except the ones in features_to_keep
        df_preprocessed = keep_features(df_preprocessed, features)  

        # Scale features
        #scaler = MinMaxScaler()
        #df_scaled = pd.DataFrame(scaler.fit_transform(df_preprocessed), columns=df_preprocessed.columns)

        #return df_scaled
        return df_preprocessed
    else:
        return pd.DataFrame()
    

df = pd.read_csv('Congestion_Analysis/4_kpi_not_scale_mr_std_dataset/scale_5/all/level_1.csv', encoding='utf-8-sig')
# Step 1: Separate the label column
X = df.drop(columns=['label'])
y = df['label']

pre_df = preprocess_dataset(X)

pre_df['label'] = y

print(pre_df)

pre_df.to_csv('DATASETS/CDR-MLC/scale_5/Short/level_1.csv', index=False)





      pRetran  sMeanPktSz  SrcRetra  PCRatio    SrcWin  SrcLoss     DstRate  \
0         0.0  112.625000       0.0     -0.0   61440.0      0.0   80.957237   
1         0.0  123.000000       0.0     -0.0   62592.0      0.0   38.852554   
2         0.0  113.714287       0.0     -0.0   64128.0      0.0   45.044174   
3         0.0  108.375000       0.0     -0.0   61568.0      0.0   55.992386   
4         0.0  108.375000       0.0     -0.0   64128.0      0.0   52.618076   
...       ...         ...       ...      ...       ...      ...         ...   
1426      0.0   67.733330       0.0     -0.0   83584.0      0.0  100.447777   
1427      0.0   68.041664       0.0     -0.0  100352.0      0.0  102.492439   
1428      0.0   70.000000       0.0     -0.0   64256.0      0.0    0.000000   
1429      0.0   66.000000       0.0     -0.0   64256.0      0.0    0.000000   
1430      0.0  168.399994       0.0     -0.0   64128.0      0.0   32.454887   

           SrcLoad          Load       DstLoad  ...

## CDR-MLC Long

In [2]:
import os
import pandas as pd
import numpy as np
from sklearn.impute import SimpleImputer
import glob

def keep_features(df, features_to_keep):
    """
    Drop all columns from the DataFrame except for the specified features.
    
    Parameters:
    - df: pd.DataFrame, the input DataFrame
    - features_to_keep: list, list of column names to retain
    
    Returns:
    - pd.DataFrame with only the specified columns
    """
    # Ensure that the features_to_keep are in the DataFrame
    features_to_keep = [feature for feature in features_to_keep if feature in df.columns]
    
    # Return a DataFrame with only the specified features
    return df[features_to_keep]

def preprocess_dataset(df):
    """
    Preprocess a single dataset by handling missing values and keeping specific features.
    
    Parameters:
    - df: pd.DataFrame, the input DataFrame
    
    Returns:
    - pd.DataFrame, preprocessed DataFrame
    """
    # Drop columns that contain only missing values
    df = df.dropna(axis=1, how='all')
    
    # Separate numeric and non-numeric columns
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    non_numeric_cols = df.select_dtypes(exclude=[np.number]).columns
    
    # Check if DataFrame has either numeric or non-numeric columns
    if not numeric_cols.empty or not non_numeric_cols.empty:
        # Handle missing values for numeric data
        if not numeric_cols.empty:
            imputer_numeric = SimpleImputer(strategy='mean')
            df_numeric = pd.DataFrame(imputer_numeric.fit_transform(df[numeric_cols]), 
                                      columns=numeric_cols)
        else:
            df_numeric = pd.DataFrame()  # Empty DataFrame for numeric data if no numeric columns exist
        
        # Concatenate processed numeric and encoded non-numeric data
        df_preprocessed = pd.concat([df_numeric], axis=1)
        
        # List of features to keep
        features = [
            'pRetran','sMeanPktSz', 'SrcRetra', 'PCRatio', 
            'SrcWin', 'SrcLoss', 'DstRate', 'SrcLoad', 'Load', 'DstLoad', 'TcpRtt', 'Sum', 'AckDat', 'dTtl', 'Min', 'Max', 'pLoss', 'DstLoss',
            'Loss', 'StdDev', 'Rate', 'SrcRate', 'IdleTime', 'Dur', 'SrcPkts',
            'SrcGap', 'DstBytes', 'DstGap', 'sTtl', 'DstWin', 'TotPkts', 'DstPkts', 'Mean',
            'SrcBytes', 'TotBytes', 'dMeanPktSz', 'DstRetra', 'SynAck'
        ]

        # Drop all columns except the ones in features_to_keep
        df_preprocessed = keep_features(df_preprocessed, features)  

        return df_preprocessed
    else:
        return pd.DataFrame()

def process_flow_files(directory_path, output_filename='CDR-MLC-long.csv'):
    """
    Process all .flow files in a directory and save to a single CSV file.
    
    Parameters:
    - directory_path: str, path to directory containing .flow files
    - output_filename: str, name of output CSV file
    
    Returns:
    - None (saves processed data to CSV file)
    """
    # Find all .flow files in the directory
    flow_files = glob.glob(os.path.join(directory_path, '*.flow'))
    
    if not flow_files:
        print(f"No .flow files found in directory: {directory_path}")
        return
    
    print(f"Found {len(flow_files)} .flow files to process")
    
    # List to store all processed DataFrames
    all_dataframes = []
    
    # Process each file
    for i, file_path in enumerate(flow_files, 1):
        try:
            print(f"Processing file {i}/{len(flow_files)}: {os.path.basename(file_path)}")
            
            # Read the file
            df = pd.read_csv(file_path, encoding='utf-8-sig')
            
            # Remove existing 'label' column if it exists
            if 'Label' in df.columns:
                df = df.drop(columns=['Label'])
                print(f"  Removed existing 'label' column")
            
            # Preprocess features
            pre_df = preprocess_dataset(df)
            
            # Check if preprocessing returned valid data
            if pre_df.empty:
                print(f"  Warning: No data after preprocessing for {file_path}")
                continue
            
            # Get filename without extension for label
            filename = os.path.basename(file_path)
            filename_without_ext = os.path.splitext(filename)[0]
            
            # Add new 'label' column with filename
            pre_df['label'] = filename_without_ext
            
            # Add to list
            all_dataframes.append(pre_df)
            
            print(f"  Processed {len(pre_df)} rows, label: {filename_without_ext}")
            
        except Exception as e:
            print(f"Error processing {file_path}: {str(e)}")
            continue
    
    if not all_dataframes:
        print("No data was successfully processed.")
        return
    
    # Concatenate all DataFrames
    print("\nConcatenating all processed data...")
    final_df = pd.concat(all_dataframes, ignore_index=True)
    
    # Save to CSV
    output_path = os.path.join(directory_path, output_filename)
    final_df.to_csv(output_path, index=False)
    
    print(f"\nProcessing complete!")
    print(f"Total rows processed: {len(final_df)}")
    print(f"Total columns: {len(final_df.columns)}")
    print(f"Output saved to: {output_path}")
    
    # Display summary
    print("\nSummary of processed data:")
    print(f"Shape: {final_df.shape}")
    print(f"Columns: {list(final_df.columns)}")
    print(f"Label distribution (by filename):")
    print(final_df['label'].value_counts())
    
    return final_df

# Example usage
if __name__ == "__main__":
    # Specify the directory containing .flow files
    directory_path = 'DATASETS/CDR-MLC/scale_1/Short/'
    
    # Process all .flow files in the directory
    processed_data = process_flow_files(directory_path, output_filename='CDR-MLC-long.csv')
    
    # Display first few rows
    if processed_data is not None:
        print("\nFirst 5 rows of processed data:")
        print(processed_data.head())

No .flow files found in directory: DATASETS/CDR-MLC/scale_1/Short/


## SDN DDOS

In [5]:
import pandas as pd

df = pd.read_csv('DATASETS/AdDDoSDN/adddosdn_cicflow_dataset.csv')

if 'Label_multi' in df.columns:
    df = df.rename(columns={'Label_multi': 'label'})

columns_to_drop = ['src_ip', 'dst_ip', 'src_port', 'dst_port', 'protocol', 'timestamp', 'Label_binary', 'Attack_Type']

existing_columns = [col for col in columns_to_drop if col in df.columns]
df_cleaned = df.drop(columns=existing_columns)

df_cleaned = df_cleaned.fillna(0)

df_cleaned.to_csv('DATASETS/AdDDoSDN/clean_standard_ddos.csv', index=False)

print(f"Dropped columns: {existing_columns}")
print(f"Remaining columns: {len(df_cleaned.columns)}")
print(f"Columns in cleaned dataset: {df_cleaned.columns.tolist()}")
print("File saved as 'clean_standard_ddos.csv'")

Dropped columns: ['src_ip', 'dst_ip', 'src_port', 'dst_port', 'protocol', 'timestamp', 'Label_binary', 'Attack_Type']
Remaining columns: 78
Columns in cleaned dataset: ['Unnamed: 0', 'flow_duration', 'flow_byts_s', 'flow_pkts_s', 'fwd_pkts_s', 'bwd_pkts_s', 'tot_fwd_pkts', 'tot_bwd_pkts', 'totlen_fwd_pkts', 'totlen_bwd_pkts', 'fwd_pkt_len_max', 'fwd_pkt_len_min', 'fwd_pkt_len_mean', 'fwd_pkt_len_std', 'bwd_pkt_len_max', 'bwd_pkt_len_min', 'bwd_pkt_len_mean', 'bwd_pkt_len_std', 'pkt_len_max', 'pkt_len_min', 'pkt_len_mean', 'pkt_len_std', 'pkt_len_var', 'fwd_header_len', 'bwd_header_len', 'fwd_seg_size_min', 'fwd_act_data_pkts', 'flow_iat_mean', 'flow_iat_max', 'flow_iat_min', 'flow_iat_std', 'fwd_iat_tot', 'fwd_iat_max', 'fwd_iat_min', 'fwd_iat_mean', 'fwd_iat_std', 'bwd_iat_tot', 'bwd_iat_max', 'bwd_iat_min', 'bwd_iat_mean', 'bwd_iat_std', 'fwd_psh_flags', 'bwd_psh_flags', 'fwd_urg_flags', 'bwd_urg_flags', 'fin_flag_cnt', 'syn_flag_cnt', 'rst_flag_cnt', 'psh_flag_cnt', 'ack_flag_cnt', 

## SDN TCP SYN

In [4]:
import pandas as pd

df = pd.read_csv('DATASETS/SDN-TCP-SYN-DDOS/TCP-SYNC-DDOS.csv')

if 'Label' in df.columns:
    df = df.rename(columns={'Label': 'label'})

columns_to_drop = ['Unnamed: 0','Flow ID','Src IP','Src Port','Dst IP','Dst Port','Protocol','Timestamp']

existing_columns = [col for col in columns_to_drop if col in df.columns]
df_cleaned = df.drop(columns=existing_columns)

df_cleaned = df_cleaned.fillna(0)

df_cleaned.to_csv('DATASETS/SDN-TCP-SYN-DDOS/clean_standard_ddos.csv', index=False)

print(f"Dropped columns: {existing_columns}")
print(f"Remaining columns: {len(df_cleaned.columns)}")
print(f"Total NaN values after replacement: {df_cleaned.isna().sum().sum()}")
print("File saved as 'DATASETS/AdDDoSDN/Clean-SYN-TCP-DDOS.csv'")

Dropped columns: ['Flow ID', 'Src IP', 'Src Port', 'Dst IP', 'Dst Port', 'Protocol', 'Timestamp']
Remaining columns: 77
Total NaN values after replacement: 0
File saved as 'DATASETS/AdDDoSDN/Clean-SYN-TCP-DDOS.csv'
